In [ ]:
import time
import json
import csv
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
import os
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# Cria o navegador e salva o login em cache
dir_path = os.getcwd()
profile = os.path.join(dir_path, "profile", "wpp")
options = webdriver.ChromeOptions()
options.add_argument(
   r"user-data-dir={}".format(profile))


# Carregar informações do arquivo JSON
with open('dados.json') as json_file:
    dados_json = json.load(json_file)

# Extraindo informações da tarefa
msg = dados_json.get('mensagem')
numeros = dados_json.get('contatos')


def wpp_browser():
    # Define o diretório do perfil
    profile_directory = os.getcwd()  # Escolha o nome do diretório onde o cache será armazenado

    # Verifica se o diretório do perfil existe, se não, o cria
    if not os.path.exists(profile_directory):
        os.makedirs(profile_directory)

    # Cria o navegador e salva o login em cache
    options = webdriver.ChromeOptions()
    options.add_argument("user-data-dir={}".format(os.path.abspath(profile_directory)))

    # Inicializar o navegador
    browser = webdriver.Chrome(options=options)
    browser.maximize_window()

    # URL do WhatsApp Web
    whatsapp_url = "https://web.whatsapp.com/"
    browser.get(whatsapp_url)

    # Verifica se a página está carregada pela primeira vez
    first_load = True

    # Verifica se o QR code está presente somente na primeira carga da página
    while first_load:
        # Verifica se o QR code está presente
        if len(browser.find_elements(By.XPATH, '//*[@id="app"]/div/div[2]/div[3]/div[1]/div/div/div[2]/div/canvas')) < 1:
            print("QR code não encontrado. Continuando sem autenticação.")
            break  # Se o QR code não estiver presente, sai do loop

        # Se o QR code estiver presente, espera até que o painel lateral seja carregado
        WebDriverWait(browser, 30).until(EC.presence_of_element_located((By.ID, 'side')))  
        first_load = False

    return browser

# Processo Span
def enviar_mensagem(browser, numeros, msg):
    # Enviar mensagem para cada contato na lista 'numeros'
    for numero in numeros:
        # Enviar mensagem para o contato
        search_box = browser.find_element(By.XPATH, "//div[@contenteditable='true']")
        search_box.send_keys(numero)
        time.sleep(0.5)  # Aguarde para garantir que o contato seja carregado
        browser.find_element(By.XPATH, "//span[text()='{}']".format(numero))
        browser.click().click()  # Clicar no contato
        # Digitar a mensagem
        message_box = browser.find_element(By.XPATH, "//div[@contenteditable='true'][@data-tab='1']")
        message_box.send_keys(msg, Keys.ENTER)



''' # Enviando cada parágrafo da mensagem
    for message in mensagem:
        message_box = driver.find_element(By.XPATH, "//div[@contenteditable='true'][@data-tab='1']")
        message_box.send_keys(message)
        message_box.send_keys(Keys.SHIFT, Keys.ENTER)

    message_box.send_keys(Keys.ENTER)'''


# Chamar a função para configurar o navegador com o cache do WhatsApp Web
browser = wpp_browser()
    
    
    

In [180]:
# Entra no grupo
#browser.find_element('xpath', '//*[@id="side"]/div[1]/div/div[2]/button/div[2]/span').click # Clica na lupa
# search_box = browser.find_element(By.XPATH, "//div[@contenteditable='true']").send_keys(nome_grupo, Keys.Enter)
#browser.find_element('xpath', '//*[@id="side"]/div[1]/div/div[2]/div[2]/div/div[1]/p').send_keys(nome_grupo)                      
#browser.find_element('xpath', '//*[@id="side"]/div[1]/div/div[2]/div[2]/div/div[1]/p').send_keys(Keys.ENTER) 
time.sleep(0.3)


In [181]:
# Localizando os contatos do grupo
contacts = browser.find_elements(By.XPATH, ".//span[@class='l7jjieqr ajgl1lbb edeob0r2 _11JPr']")       

# Extraindo e exibindo os contatos
print("Contatos no grupo:")
for contact in contacts:
    print(contact.text)

# Salvando os contatos em um arquivo CSV
csv_file_path = 'contatos.csv'

with open(csv_file_path, 'w', newline='', encoding='utf-8') as file:
    csv_writer = csv.writer(file)
    # csv_writer.writerow(['Contatos : ' + nome_grupo])  # Escrever o cabeçalho do CSV

    # Iterar sobre os contatos e salvá-los no arquivo CSV
    for contact in contacts:
        contact_text = contact.text
        csv_writer.writerow([contact_text])



In [194]:

# Ler os contatos do arquivo CSV
csv_file_path = 'contatos.csv'
with open(csv_file_path, 'r', newline='', encoding='utf-8') as file:
        reader = csv.reader(file)
        next(reader)  # Pule o cabeçalho do CSV
        contatos = [row[0] for row in reader]

    # Iterar sobre os contatos e enviar mensagens
mensagem = msg

for contato in contatos:
        # Voltar para o grupo
        search_box = browser.find_element(By.XPATH, "//div[@contenteditable='true']")
        search_box.send_keys(Keys.ENTER)
        
        # Aguardar um momento para garantir que o grupo seja carregado
        time.sleep(2)

        # Enviar mensagem para o contato
        enviar_mensagem(browser, contato, msg)
        
        # Aguardar um momento antes de passar para o próximo contato
        time.sleep(2)